In [1]:
# run libraries
import pandas as pd
import numpy as np
from datetime import datetime # asdf

In [2]:
# read and load data files
stops = pd.read_csv("data/stops.txt")
routes = pd.read_csv("data/routes.txt")
stop_times = pd.read_csv("data/stop_times.txt")
trips = pd.read_csv("data/trips.txt")
calendar = pd.read_csv("data/calendar.txt")
calendar_dates = pd.read_csv("data/calendar_dates.txt")
shapes = pd.read_csv("data/shapes.txt")

In [3]:
# helper function to read and validate schema for each data set
def schema(dataframe):
    print(dataframe.shape, "\n\nData Types:")
    print(dataframe.dtypes, "\n\nMissing Entries:")
    print(dataframe.isnull().sum())

---

## Table of Contents
1. [**Stops**](#stops)
2. [**Routes**](#routes)
3. [**Stop Times**](#stop-times)
4. [**Trips**](#trips)
5. [**Calendar**](#calendar)
6. [**Calendar Dates**](#calendar-dates)
7. [**Shapes**](#shapes)
8. [**Foreign Key Validations**](#foreign-key-validations)
9. [**Edge List**](#edge-list)

---

## Stops

The `stops` data tells us information regarding transit stopping points across Calgary. 

Includes the following columns:
- `stop_id`: Identifies a location to stop at: stop/platform, station, entrance/exit, generic node or boarding area
- `stop_code`: Short text or a number that identifies the location for riders. These codes are often used in phone-based transit information systems or printed on signage to make it easier for riders to get information for a particular location. May be the same as `stop_id`.
- `stop_name`: Name of the location.
- `stop_desc`: Description of the location that provides useful, quality information.
- `stop_lat`: Latitude of the location.
- `stop_lon`: Longitude of the location.
- `zone_id`: Identifies the fare zone for a stop. If this record represents a station or station entrance, the `zone_id` is ignored.
- `stop_url`: URL of a web page about the location.
- `location_type`: Type of location, classifed as:
    - `0` (or empty) - **Stop** (or **Platform**). A location where passengers board or disembark from a transit vehicle.
    - `1` - **Station**. A physical structure or area that contains one or more platform.
    - `2` - **Entrance/Exit**. A location where passengers can enter or exit a station from the street.
    - `3` - **Generic Node**. A location within a station, not matching any other location_type.
    - `4` - **Boarding Area**. A specific location on a platform, where passengers can board and/or dropoff from vehicles.


*For further information, refer to the official GTFS documentation (https://gtfs.org/documentation/schedule/reference/#stopstxt).*

In [4]:
# schema and data validation
schema(stops)
stops.head()

(6173, 9) 

Data Types:
stop_id            int64
stop_code          int64
stop_name         object
stop_desc        float64
stop_lat         float64
stop_lon         float64
zone_id          float64
stop_url         float64
location_type      int64
dtype: object 

Missing Entries:
stop_id             0
stop_code           0
stop_name           0
stop_desc        6173
stop_lat            0
stop_lon            0
zone_id          6173
stop_url         6173
location_type       0
dtype: int64


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type
0,112112,112112,Crescent Heights High School,NaN,51.060931,-114.065158,NaN,NaN,0
1,1800,1800,EB 17 AV SE @ 84 ST SE,NaN,51.037651,-113.911093,NaN,NaN,0
2,1802,1802,NB Centre ST N @ 78 AV N nearside,NaN,51.122337,-114.071250,NaN,NaN,0
3,1803,1803,EB 46 AV SE @ Highfield CI SE,NaN,51.011541,-114.044189,NaN,NaN,0
4,1804,1804,EB 120 AV NE @ 16 ST NE,NaN,51.161538,-114.022383,NaN,NaN,0


In [5]:
# remove 'zone_id', 'stop_desc', and 'stop_url' columns since they are all empty columns and provide no useful information for later
stops = stops.drop(['stop_code', 'zone_id', 'stop_desc', 'stop_url'], axis=1)
stops

,stop_id,stop_name,stop_lat,stop_lon,location_type
0,112112,Crescent Heights High School,51.060931,-114.065158,0
1,1800,EB 17 AV SE @ 84 ST SE,51.037651,-113.911093,0
2,1802,NB Centre ST N @ 78 AV N nearside,51.122337,-114.071250,0
3,1803,EB 46 AV SE @ Highfield CI SE,51.011541,-114.044189,0
4,1804,EB 120 AV NE @ 16 ST NE,51.161538,-114.022383,0
...,...,...,...,...,...
6168,9983,WB 6 AV SW @ E. of 1 ST SW,51.047627,-114.064416,0
6169,9984,WB 6 AV SE @ 1 ST SE,51.047498,-114.060015,0
6170,9995,EB Dalhousie DR @ Dalham CR NW,51.106221,-114.158316,0
6171,9996,NB 52 ST SE @ 86 AV SE,50.976257,-113.958240,0


In [6]:
# helper function to check for duplicates (for id columns)
def unique_column(df, col_name):
    counts = df[col_name].value_counts()
    dupes = counts[counts > 1]
    return dupes.index.tolist()

In [7]:
# verify invalid entries
# ensure all stop_ids are unique
print("Duplicate stop_ids:", unique_column(stops, 'stop_id'))

# check for any invalid 'location_type' codes
invalid_location_type = stops[~stops['location_type'].isin([0,1,2,3,4])]['stop_id'].tolist()
print("Stops with invalid location_type:", invalid_location_type) 

Duplicate stop_ids: []
Stops with invalid location_type: []


---

## Routes

The `routes` data tells us information regarding transit routes across Calgary. 

Includes the following columns:
- `route_id`: Identifies a route.
- `route_short_name`: Short name of a route. Often a short, abstract identifier (e.g., "32", "100X", "Green") that riders use to identify a route.
- `route_long_name`: Full name of a route. This name is generally more descriptive than the route_short_name and often includes the route's destination or stop.
- `route_desc`: Description of a route that provides useful, quality information.
- `route_type`: Indicates the type of transportation used on a route. These are labeled as:
    - `0` - **Tram, Streetcar, Light rail**. Any light rail or street level system within a metropolitan area.
    - `1` - **Subway, Metro**. Any underground rail system within a metropolitan area.
    - `2` - **Rail**. Used for intercity or long-distance travel.
    - `3` - **Bus**. Used for short- and long-distance bus routes.
    - `4` - **Ferry**. Used for short- and long-distance boat service.
    - `5` - **Cable Tram**. Used for street-level rail cars where the cable runs beneath the vehicle (e.g., cable car in San Francisco).
    - `6` - **Aerial lift, suspended cable car (e.g., gondola lift, aerial tramway)**. Cable transport where cabins, cars, gondolas or open chairs are suspended by means of one or more cables.
    - `7` - **Funicular**. Any rail system designed for steep inclines.
    - `11` - **Trolleybus**. Electric buses that draw power from overhead wires using poles.
    - `12` - **Monorail**. Railway in which the track consists of a single rail or a beam.
- `route_url`: URL of a web page about the particular route.
- `route_color`: Route color designation that matches public facing material. Defaults to white (`#FFFFFF`) when omitted or left empty.
- `route_text_color`: Legible color to use for text drawn against a background of route_color. Defaults to black (`#000000`) when omitted or left empty.

*For further information, refer to the official GTFS documentation (https://gtfs.org/documentation/schedule/reference/#routestxt).*

In [8]:
schema(routes)
routes.head()

(414, 8) 

Data Types:
route_id             object
route_short_name     object
route_long_name      object
route_desc          float64
route_type            int64
route_url           float64
route_color         float64
route_text_color    float64
dtype: object 

Missing Entries:
route_id              0
route_short_name      0
route_long_name       0
route_desc          414
route_type            0
route_url           414
route_color         414
route_text_color    414
dtype: int64


,route_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,1-20781,1,Bowness/Forest Lawn,NaN,3,NaN,NaN,NaN
1,1-20789,1,Bowness/Forest Lawn,NaN,3,NaN,NaN,NaN
2,2-20781,2,Mount Pleasant/Killarney 17 Av SW,NaN,3,NaN,NaN,NaN
3,2-20789,2,Mount Pleasant/Killarney 17 Av SW,NaN,3,NaN,NaN,NaN
4,3-20781,3,Sandstone/Elbow Dr SW,NaN,3,NaN,NaN,NaN


In [9]:
# remove 'route_desc', 'route_url', 'route_color' and 'route_text_color' as they are all empty and contain no useful information
routes = routes.drop(['route_desc', 'route_url', 'route_color', 'route_text_color'], axis=1)
routes

,route_id,route_short_name,route_long_name,route_type
0,1-20781,1,Bowness/Forest Lawn,3
1,1-20789,1,Bowness/Forest Lawn,3
2,2-20781,2,Mount Pleasant/Killarney 17 Av SW,3
3,2-20789,2,Mount Pleasant/Killarney 17 Av SW,3
4,3-20781,3,Sandstone/Elbow Dr SW,3
...,...,...,...,...
409,890-20781,890,St. John Henry Newman/ Seton,3
410,892-20781,892,St. Isabella/ Mckenzie,3
411,894-20781,894,St. Gregory/ 69 St SW/Strathcona,3
412,895-20781,895,St. Gregory/ West Springs/ CougarRidge,3


In [10]:
# verify any invalid entries 
# ensure all route ids are unique
print("Duplicate route_ids:", unique_column(routes, 'route_id'))

# check 'route_types' codes
invalid_route_type = routes[~routes['route_type'].isin([0,1,2,3,4,5,6,7,11,12])]['route_id'].tolist()
print("Trips with invalid route_type:", invalid_route_type) 

Duplicate route_ids: []
Trips with invalid route_type: []


---

## Stop Times

The `stop_times` data tells us information regarding transit stop times across Calgary. 

Includes the following columns:
- `trip_id`: **Identifies a trip**. References `trip_id` from `trips.txt`
- `arrival_time`: **Arrival time at the stop for a specific trip**. If there are not separate times for arrival and departure at a stop, `arrival_time` and `departure_time` should be the same. For times occurring after midnight on the service day, the time is entered as a value greater than 24:00:00 in HH:MM:SS.
- `departure_time`: **Departure time from the stop for a specific tri**p. For times occurring after midnight on the service day, the time is entered as a value greater than 24:00:00 in HH:MM:SS.
- `stop_id`: **Identifies the serviced stop**. A stop may be serviced multiple times in the same trip, and multiple trips and routes may service the same stop. References `stop_id` from `stops.txt`
- `stop_sequence`: **Order of stops, location groups, or GeoJSON locations for a particular trip**. The values must increase along the trip but do not need to be consecutive.
- `pickup_type`: **Indicates pickup method**. Valid options are:
    - `0` or empty - Regularly scheduled pickup.
    - `1` - No pickup available.
    - `2` - Must phone agency to arrange pickup.
    - `3` - Must coordinate with driver to arrange pickup.
- `drop_off_type`: **Indicates drop off method**. Valid options are:
    - `0` or empty - Regularly scheduled drop off.
    - `1` - No drop off available.
    - `2` - Must phone agency to arrange drop off.
    - `3` - Must coordinate with driver to arrange drop off.
- `shape_dist_traveled`: **Actual distance traveled along the associated shape, from the first stop to the stop specified in this record**. This field specifies how much of the shape to draw between any two stops during a trip. Values used for `shape_dist_traveled` must increase along with the `stop_sequence` field; they must not be used to show reverse travel along a route.
- `timepoint`: **Indicates if arrival and departure times for a stop are strictly adhered to by the vehicle or if they are instead approximate and/or interpolated times**. Valid options are:
    - `0` - Times are considered approximate.
    - `1` - Times are considered exact.

*For further information, refer to the official GTFS documentation (https://gtfs.org/documentation/schedule/reference/#stop_timestxt).*

In [11]:
schema(stop_times)
stop_times.head()

(3699315, 9) 

Data Types:
trip_id                  int64
arrival_time            object
departure_time          object
stop_id                  int64
stop_sequence            int64
pickup_type              int64
drop_off_type            int64
shape_dist_traveled    float64
timepoint                int64
dtype: object 

Missing Entries:
trip_id                0
arrival_time           0
departure_time         0
stop_id                0
stop_sequence          0
pickup_type            0
drop_off_type          0
shape_dist_traveled    0
timepoint              0
dtype: int64


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,shape_dist_traveled,timepoint
0,74468238,11:52:00,11:52:00,6451,1,0,0,0.000,1
1,74468238,11:53:00,11:53:00,6712,2,0,0,0.557,0
2,74468238,11:53:00,11:53:00,7276,3,0,0,0.866,0
3,74468238,11:59:00,11:59:00,8058,4,0,0,5.359,1
4,74468238,12:00:00,12:00:00,6338,5,0,0,5.659,0


In [12]:
# drop columns we will not be using for this project
stop_times = stop_times.drop(['departure_time', 'pickup_type', 'drop_off_type', 'timepoint', 'shape_dist_traveled'], axis=1)
stop_times

,trip_id,arrival_time,stop_id,stop_sequence
0,74468238,11:52:00,6451,1
1,74468238,11:53:00,6712,2
2,74468238,11:53:00,7276,3
3,74468238,11:59:00,8058,4
4,74468238,12:00:00,6338,5
...,...,...,...,...
3699310,74828111,22:27:00,6809,12
3699311,74828111,22:29:00,9385,13
3699312,74828111,22:32:00,9896,14
3699313,74828111,22:34:00,9396,15


In [13]:
# fix timeformat of stop_times --> convert to seconds since midnight (of the referenced day)
def seconds_from_midnight(timestamp):
    h,m,s = map(int, timestamp.split(":"))
    return int(h*3600 + m*60 + s)

stop_times['arrival_time'] = stop_times['arrival_time'].apply(seconds_from_midnight)
stop_times.head()

,trip_id,arrival_time,stop_id,stop_sequence
0,74468238,42720,6451,1
1,74468238,42780,6712,2
2,74468238,42780,7276,3
3,74468238,43140,8058,4
4,74468238,43200,6338,5


explanation of sorting the stop_times for edge list

In [14]:
# sort to group trip_ids together, and resort stop_sequences 
stop_times = stop_times.sort_values(by=['trip_id', 'stop_sequence'])
stop_times.head(30)

,trip_id,arrival_time,stop_id,stop_sequence
3622952,72471336,17640,3641,1
3622953,72471336,17820,3817,2
3622954,72471336,18060,3960,3
3622955,72471336,18240,6815,4
3622956,72471336,18360,8566,5
3622957,72471336,18480,8565,6
3622958,72471336,18660,8564,7
3622959,72471336,18780,8563,8
3622960,72471336,18900,8562,9
3622961,72471336,19200,6827,10


In [15]:
# verify any invalid entries 
# ensure 'arrival_time' have no negative or invalid entries
invalid_arrival_times = stop_times[stop_times['arrival_time'] < 0]['trip_id'].tolist()
print("Trips with invalid arrival times:", invalid_arrival_times) 

# ensure stop_sequence is in increasing order for each distinct trip (checked under trip_id)
invalid_stop_sequences = (stop_times.groupby('trip_id')['stop_sequence'].apply(lambda x: not (x.is_monotonic_increasing and x.is_unique)))
print(f"Trips with invalid stop_sequence: {invalid_stop_sequences[invalid_stop_sequences].index.tolist()}") 

Trips with invalid arrival times: []
Trips with invalid stop_sequence: []


---

## Trips

The `trips` data tells us information regarding trips that transits will take across Calgary. 

Includes the following columns:
- `route_id`: **Identifies a route**.
- `service_id`: **Identifies a set of dates when service is available for one or more routes**.
- `trip_id`: **Identifies a trip**.
- `trip_headsign`: **Text that appears on signage identifying the trip's destination to riders**. This field is recommended for all services with headsign text displayed on the vehicle which may be used to distinguish amongst trips in a route.
- `direction_id`: **Indicates the direction of travel for a trip**. This field should not be used in routing; it provides a way to separate trips by direction when publishing time tables. Valid options are:
    - `0`: Travel in one direction (e.g. outbound travel).
    - `1`: Travel in the opposite direction (e.g. inbound travel).
- `block_id`: **Identifies the block to which the trip belongs**. A block consists of a single trip or many sequential trips made using the same vehicle, defined by shared service days and `block_id`. A `block_id` may have trips with different service days, making distinct blocks. 
- `shape_id`: **Identifies a geospatial shape describing the vehicle travel path for a trip**.

*For further information, refer to the official GTFS documentation (https://gtfs.org/documentation/schedule/reference/#tripstxt).*

In [16]:
schema(trips)
trips.head()

(111146, 7) 

Data Types:
route_id         object
service_id       object
trip_id           int64
trip_headsign    object
direction_id      int64
block_id          int64
shape_id          int64
dtype: object 

Missing Entries:
route_id         0
service_id       0
trip_id          0
trip_headsign    0
direction_id     0
block_id         0
shape_id         0
dtype: int64


,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id
0,36-20789,2026JU-1BUSSA-Saturday-02,74468238,RIVERBEND,0,7144090,360037
1,36-20789,2026JU-1BUSSA-Saturday-02,74468239,RIVERBEND,0,7144090,360037
2,36-20789,2026JU-1BUSSA-Saturday-02,74468240,RIVERBEND,0,7144090,360037
3,36-20789,2026JU-1BUSSA-Saturday-02,74468241,RIVERBEND,0,7144090,360037
4,36-20789,2026JU-1BUSSA-Saturday-02,74468242,RIVERBEND,0,7144090,360037


In [17]:
# drop columns we will not be using for this project
trips = trips.drop(['trip_headsign', 'block_id'], axis=1)
trips

,route_id,service_id,trip_id,direction_id,shape_id
0,36-20789,2026JU-1BUSSA-Saturday-02,74468238,0,360037
1,36-20789,2026JU-1BUSSA-Saturday-02,74468239,0,360037
2,36-20789,2026JU-1BUSSA-Saturday-02,74468240,0,360037
3,36-20789,2026JU-1BUSSA-Saturday-02,74468241,0,360037
4,36-20789,2026JU-1BUSSA-Saturday-02,74468242,0,360037
...,...,...,...,...,...
111141,202-20780,2026MA-pT7Jun14-Sunday-01,74828106,1,2020435
111142,202-20780,2026MA-pT7Jun14-Sunday-01,74828108,1,2020435
111143,202-20780,2026MA-pT7Jun14-Sunday-01,74828109,0,2020436
111144,202-20780,2026MA-pT7Jun14-Sunday-01,74828110,1,2020435


In [18]:
# verify any invalid entries 
# ensure all trip_ids are unique
print("Duplicate trip_ids:", unique_column(trips, 'trip_id'))

# 'direction_id' codes
invalid_direction_ids = trips[~trips['direction_id'].isin([0,1])]['trip_id'].tolist()
print("Trips with invalid direction_id:", invalid_direction_ids)

Duplicate trip_ids: []
Trips with invalid direction_id: []


---

## Calendar

The `calendar` data tells us information regarding service operations for transit routes across Calgary. 

Includes the following columns:
- `service_id`: **Identifies a set of dates when service is available for one or more routes**.
- `monday`: **Indicates whether the service operates on all Mondays in the date range specified by the `start_date` and ``end_date`` fields**. 
    - `1` - Service is available for all Mondays in the date range.
    - `0` - Service is not available for Mondays in the date range.
- `tuesday`: Functions in the same way as `monday` except applies to Tuesdays
- `wednesday`: Functions in the same way as `monday` except applies to Wednesdays
- `thursday`: Functions in the same way as `monday` except applies to Thursdays
- `friday`: Functions in the same way as `monday` except applies to Fridays
- `saturday`: Functions in the same way as `monday` except applies to Saturdays
- `sunday`: Functions in the same way as `monday` except applies to Sundays
- `start_date`: **Starting service day for the service interval**.
- `end_date`: **End service day for the service interval**. This service day is included in the interval.

*For further information, refer to the official GTFS documentation (https://gtfs.org/documentation/schedule/reference/#calendartxt).*

In [19]:
schema(calendar)
calendar.head()

(33, 10) 

Data Types:
service_id    object
monday         int64
tuesday        int64
wednesday      int64
thursday       int64
friday         int64
saturday       int64
sunday         int64
start_date     int64
end_date       int64
dtype: object 

Missing Entries:
service_id    0
monday        0
tuesday       0
wednesday     0
thursday      0
friday        0
saturday      0
sunday        0
start_date    0
end_date      0
dtype: int64


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,2026MA-1BUSWK-Weekday-03,1,1,1,1,1,0,0,20260603,20260619
1,2026MA-1BUSWK-Weekday-03-1111000,1,1,1,1,0,0,0,20260603,20260619
2,2026MA-1BUSWK-Weekday-03-0000100,0,0,0,0,1,0,0,20260603,20260619
3,2026MA-1BUSSA-Saturday-03,0,0,0,0,0,1,0,20260606,20260620
4,2026MA-1BUSSU-Sunday-03,0,0,0,0,0,0,1,20260607,20260621


In [20]:
# convert start_date and end_date to proper datetime format
calendar['start_date'] = calendar['start_date'].apply(lambda row: datetime.strptime(str(row), "%Y%m%d"))
calendar['end_date'] = calendar['end_date'].apply(lambda row: datetime.strptime(str(row), "%Y%m%d"))
calendar.head()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,2026MA-1BUSWK-Weekday-03,1,1,1,1,1,0,0,2026-06-03,2026-06-19
1,2026MA-1BUSWK-Weekday-03-1111000,1,1,1,1,0,0,0,2026-06-03,2026-06-19
2,2026MA-1BUSWK-Weekday-03-0000100,0,0,0,0,1,0,0,2026-06-03,2026-06-19
3,2026MA-1BUSSA-Saturday-03,0,0,0,0,0,1,0,2026-06-06,2026-06-20
4,2026MA-1BUSSU-Sunday-03,0,0,0,0,0,0,1,2026-06-07,2026-06-21


In [21]:
# verify any invalid entries 
# 'days of the week' codes
days = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
for day in days:
    invalid_day_code = calendar[~calendar[day].isin([0,1])]['service_id'].tolist()
    print(f"Trips with invalid {day}_code:", invalid_day_code)

Trips with invalid monday_code: []
Trips with invalid tuesday_code: []
Trips with invalid wednesday_code: []
Trips with invalid thursday_code: []
Trips with invalid friday_code: []
Trips with invalid saturday_code: []
Trips with invalid sunday_code: []


In [22]:
# verify all start_dates start before end_date
invalid_dates = calendar['start_date'] > calendar['end_date']
print("Dates where start_date comes after end_date:", calendar[invalid_dates]['service_id'].tolist())

Dates where start_date comes after end_date: []


---


## Calendar Dates

The `calendar_dates` data tells us information regarding exceptions for usual service operations for transit routes across Calgary. 

Includes the following columns:
- `service_id`: **A code identifying a scheduled service pattern used to link trips to the days they run**. For example `2026JU-1BUSWK-Weekday-02`:
    - `2026JU` - represents year and month where service exception occurs (June 2026)
    - `1BUS` - represents transit service being affected (Bus)
    - `WK` - represents the day of the week where exception occurs (Weekday)
    - `Weekday-02` - reiterates day of the week and iteration/version of this exception update (reiterates Weekday, and is the 2nd version of this exception update).
- `date`: Date when service exception occurs.
- `exception_type`: **Indicates whether service is available on the date specified in the date field**. Valid options are:
   - `1` - Service has been added for the specified date.
   - `2` - Service has been removed for the specified date.

*For further information, refer to the official GTFS documentation (https://gtfs.org/documentation/schedule/reference/#calendar_datestxt).*

In [23]:
schema(calendar_dates)
calendar_dates.head()

(31, 3) 

Data Types:
service_id        object
date               int64
exception_type     int64
dtype: object 

Missing Entries:
service_id        0
date              0
exception_type    0
dtype: int64


,service_id,date,exception_type
0,2026JU-1BUSWK-Weekday-02,20260701,2
1,2026JU-1BUSWK-Weekday-02,20260702,2
2,2026JU-1BUSWK-Weekday-02,20260703,2
3,2026JU-1BUSWK-Weekday-02,20260706,2
4,2026JU-1BUSWK-Weekday-02,20260707,2


In [24]:
# convert date into proper datetime format 
from datetime import datetime

calendar_dates['date'] = calendar_dates['date'].apply(lambda row: datetime.strptime(str(row), "%Y%m%d"))
calendar_dates.head()

,service_id,date,exception_type
0,2026JU-1BUSWK-Weekday-02,2026-07-01,2
1,2026JU-1BUSWK-Weekday-02,2026-07-02,2
2,2026JU-1BUSWK-Weekday-02,2026-07-03,2
3,2026JU-1BUSWK-Weekday-02,2026-07-06,2
4,2026JU-1BUSWK-Weekday-02,2026-07-07,2


In [25]:
# verify any invalid entries 
# 'exception_type' codes
invalid_exception_codes = calendar_dates[~calendar_dates['exception_type'].isin([1,2])]['service_id'].tolist()
print(f"Trips with invalid exception_type:", invalid_exception_codes)

Trips with invalid exception_type: []


---


## Shapes

The `shapes` data tells us information describing the paths that a transit vehicle travels along its route across Calgary. Shapes are associated with Trips, and consist of a sequence of points through which the vehicle passes in order. Shapes do not need to intercept the location of Stops exactly, but all Stops on a trip should lie within a small distance of the shape for that trip, i.e. close to straight line segments connecting the shape points. 

Includes the following columns:
- `shape_id`: **Identifies a shape**.
- `shape_pt_lat`: **Latitude of a shape point**.
- `shape_pt_lon`: **Longitude of a shape point**.
- `shape_pt_sequence`: **Sequence in which the shape points connect to form the shape. Values must increase along the trip but do not need to be consecutive**.
- `shape_dist_traveled`: **Actual distance traveled along the shape from the first shape point to the point specified in this record**. Values must increase along with `shape_pt_sequence`; they must not be used to show reverse travel along a route.

*For further information, refer to the official GTFS documentation (https://gtfs.org/documentation/schedule/reference/#shapestxt).*

In [26]:
schema(shapes)
shapes.head()

(384646, 5) 

Data Types:
shape_id                 int64
shape_pt_lat           float64
shape_pt_lon           float64
shape_pt_sequence        int64
shape_dist_traveled    float64
dtype: object 

Missing Entries:
shape_id               0
shape_pt_lat           0
shape_pt_lon           0
shape_pt_sequence      0
shape_dist_traveled    0
dtype: int64


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,10180,51.052002,-113.941285,10001,0.000
1,10180,51.052017,-113.941298,10002,0.002
2,10180,51.052050,-113.941350,10003,0.007
3,10180,51.052061,-113.941391,10004,0.010
4,10180,51.052065,-113.941431,10005,0.013


In [27]:
# verify any invalid entries 
# ensure 'shape_pt_sequence' is increasing for each distinct trip (shape_id)
invalid_sequences = (shapes.groupby('shape_id')['shape_pt_sequence'].apply(lambda x: not (x.is_monotonic_increasing and x.is_unique)))
print(f"Shapes with invalid shape_pt_sequence: {invalid_sequences[invalid_sequences].index.tolist()}") 

Shapes with invalid shape_pt_sequence: []


---

## Foreign Key Validations

description

In [28]:
# key validation helper function
def validate_keys(df1, df2, id_name, df1_name, df2_name):
    key_col1 = set(df1[id_name])
    key_col2 = set(df2[id_name])

    # check if each and every element from df1 in the id_name column is the same as each element in the df2 id_name column
    if key_col1 == key_col2:
        print(f"all {id_name}s exist and are the same in both tables.")

    else:
        # determine if ids that exist from df1 do not exist in df2 
        in_df1_but_not_df2 = key_col1 - key_col2
        if in_df1_but_not_df2:
            print(f"Missing items from {df1_name} not found in {df2_name}: {sorted(in_df1_but_not_df2)}") 

        # determine if ids that exist from df2 do not exist in df1
        in_df2_but_not_df1 = key_col2 - key_col1
        if in_df2_but_not_df1:
            print(f"Missing items from {df2_name} not found in {df1_name}:  {sorted(in_df2_but_not_df1)}") 

In [29]:
# check every `stop_id` from the `stop_times` table each also all exists in the `stops` file 
validate_keys(stops, stop_times, 'stop_id', 'stops', 'stop_times') 

# check every `trip_id` from the `stop_times` table each also all exists in the `trips` file 
validate_keys(trips, stop_times, 'trip_id', 'trips', 'stop_times') 

# check every `route_id` from the `trips` table each also all exists in the `routes` file 
validate_keys(routes, trips, 'route_id', 'routes', 'trips')

all stop_ids exist and are the same in both tables.
all trip_ids exist and are the same in both tables.
all route_ids exist and are the same in both tables.


---

## Edge List

In [30]:
# create new dataframe for stop_total_times -- creating 'edges' between each stop and getting total time taken to get from one stop to the next
stop_total_times = stop_times[['trip_id', 'stop_sequence', 'arrival_time', 'stop_id']].copy()
stop_total_times['to_stop'] = stop_total_times.groupby('trip_id')['stop_id'].shift(-1) # create `to_stop` column -- stop_id of the stop the transit is heading to
stop_total_times['end_time'] = stop_total_times.groupby('trip_id')['arrival_time'].shift(-1) # create `end_time` column -- time since midnight when arriving at next stop
stop_total_times = stop_total_times.dropna(subset=['to_stop', 'end_time']) # remove NaN entries from shifting
stop_total_times = stop_total_times.astype(int) # convert entries back to int

stop_total_times = stop_total_times.rename(columns={'stop_id': 'from_stop'})
stop_total_times.head(30)  

,trip_id,stop_sequence,arrival_time,from_stop,to_stop,end_time
3622952,72471336,1,17640,3641,3817,17820
3622953,72471336,2,17820,3817,3960,18060
3622954,72471336,3,18060,3960,6815,18240
3622955,72471336,4,18240,6815,8566,18360
3622956,72471336,5,18360,8566,8565,18480
3622957,72471336,6,18480,8565,8564,18660
3622958,72471336,7,18660,8564,8563,18780
3622959,72471336,8,18780,8563,8562,18900
3622960,72471336,9,18900,8562,6827,19200
3622961,72471336,10,19200,6827,6828,19260


explain why trip time of 0 or less is bad -- only for this edge list

In [31]:
# calculate travel time
stop_total_times['trip_time'] = stop_total_times['end_time'] - stop_total_times['arrival_time']

# verify there are no negative or 0 entries for trip time (invalid entries)
invalid_trip_times = stop_total_times['trip_time'] <= 0
print("Trips with invalid trip_time:", stop_total_times[invalid_trip_times]['trip_id'].unique().tolist())

Trips with invalid trip_time: [73224970, 73224971, 73224972, 73224973, 73224974, 73224975, 73224976, 73224977, 73224978, 73224979, 73224980, 73224981, 73224982, 73224983, 73224984, 73224985, 73224986, 73224987, 73224988, 73224989, 73224990, 73224991, 73224992, 73224993, 73224994, 73224995, 73224996, 73224997, 73224998, 73224999, 73225000, 73225001, 73225002, 73225003, 73225004, 73225005, 73225006, 73225007, 73225008, 73225009, 73225010, 73225011, 73225012, 73225013, 73225014, 73225015, 73225016, 73225017, 73225018, 73225019, 73225020, 73225021, 73225022, 73225023, 73225024, 73225025, 73225026, 73225027, 73225028, 73225029, 73225030, 73225031, 73225032, 73225033, 73225034, 73225035, 73225036, 73225037, 73225038, 73225039, 73225040, 73225041, 73225042, 73225043, 73225044, 73225045, 73225046, 73225047, 73225048, 73225049, 73225050, 73225051, 73225052, 73225053, 73225054, 73225055, 73225056, 73225057, 73225058, 73225059, 73225060, 73225061, 73225062, 73225063, 73225064, 73225065, 73225066,

side note: due to HH:MM precision, the arrival_time (and thus trip_time and end_time columns) can only represent exact minutes, e.g. a value of 0 (which we will clean) actually means any amount of seconds between 0s and less than 60s (e.g. 30s becomes 0s), and 60s means any time between 60s and less than 120s (e.g. 100s becomes 60s)

In [32]:
# impute values with trip times <= 0 with 30s (reasonable approximation since it is only a few seconds between stops likely)
stop_total_times['trip_time'] = stop_total_times['trip_time'].clip(lower=0).replace(0, 30)

# recheck entries
invalid_trip_times = stop_total_times['trip_time'] <= 0
print("Trips with invalid trip_time:", stop_total_times[invalid_trip_times]['trip_id'].unique().tolist())

Trips with invalid trip_time: []


Group each trip by `from_stop` (Stop A) and `to_stop` (Stop B) and calculate the average travel time for each and every trip going from Stop A to Stop B

In [33]:
edge_list = stop_total_times.groupby(['from_stop', 'to_stop']).agg(
    number_of_trips=('trip_time', 'count'),
    average_trip_time=('trip_time', 'mean')
).reset_index()

edge_list

,from_stop,to_stop,number_of_trips,average_trip_time
0,1800,2788,646,178.142415
1,1802,5268,477,60.000000
2,1803,7747,229,120.000000
3,1804,1805,1112,60.000000
4,1805,1806,1112,47.077338
...,...,...,...,...
7567,9995,4000,1,60.000000
7568,9995,5478,1,60.000000
7569,9996,4977,858,138.531469
7570,9998,4548,2,30.000000


In [34]:
# export edge list as data 
edge_list.to_csv('data/edge_list.csv', index=False)

---

Move all cleaned data into PostgreSQL database

In [35]:
# connect engine to PostgreSQL database
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()
url = os.getenv("POSTGRES_URL")
engine = create_engine(url)

---